In [ ]:
pip install gudhi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 60.3 MB/s eta 0:00:00


In [ ]:
pip install persim

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.0 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=a23e0f9332c959468d2d0928a8c97b9468dc7a86938f758e1d8614b5da6a4ffa
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


In [ ]:
from google.colab import drive

# force_remount=True로 강제 재마운트
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
from gudhi import RipsComplex
import numpy as np
import matplotlib.pyplot as plt
from gudhi.representations import PersistenceImage
from collections import defaultdict

from persim import PersistenceImager
import persim.images_weights as weights

In [ ]:
def compute_Rips(points, max_edge=10):
    """Compute Rips complex from point cloud."""
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=2)
    return st


def divide_filtration(st):
    """Extract simplices and filtration values from simplex tree."""
    simplex_filt_pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    list_simplex = [pair[0] for pair in simplex_filt_pairs]
    list_filt = [pair[1] for pair in simplex_filt_pairs]
    return list_simplex, list_filt


def _build_boundary(simplices):
    """Build boundary matrix for a list of simplices in filtration order."""
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    boundary = []
    for s in simplices:
        if len(s) <= 1:
            boundary.append(set())
        else:
            rows = set()
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                if face in sf_to_idx:
                    rows.add(sf_to_idx[face])
            boundary.append(rows)
    return boundary


def _reduce_with_V(columns):
    """
    Left-to-right Z/2Z column reduction.
    Returns: R (reduced matrix), low (pivot positions), V (transformation matrix)
    """
    m = len(columns)
    R = [set(col) for col in columns]
    V = [{i} for i in range(m)]
    low = [-1] * m
    pivot_of_row = {}

    for i in range(m):
        while R[i]:
            li = max(R[i])
            if li in pivot_of_row:
                owner = pivot_of_row[li]
                R[i] ^= R[owner]
                V[i] ^= V[owner]
            else:
                pivot_of_row[li] = i
                low[i] = li
                break
        else:
            low[i] = -1
    return R, low, V

In [ ]:
def compute_Persistence_barcode(A):
    fil_A = compute_Rips(A)
    fil_A.persistence()

    bar_a0 = fil_A.persistence_intervals_in_dimension(0)
    bar_A0 = [bar for bar in bar_a0 if bar[1] != np.inf]
    bar_a1 = fil_A.persistence_intervals_in_dimension(1)
    bar_A1 = [bar for bar in bar_a1 if bar[1] != np.inf]

    bar0 = []
    for bar in bar_A0:
        if bar[1]-bar[0] > 1e-5:
            bar0.append([bar[0], bar[1]])
    bar1 = []
    for bar in bar_A1:
        if bar[1]-bar[0] > 1e-5:
            bar1.append([bar[0], bar[1]])

    bar_A = {}
    bar_A[0] = np.array(bar0)
    bar_A[1] = np.array(bar1)

    return bar_A

def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    keys = barcodes.keys()

    for key in keys:
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))

    vector = {}

    # =========================================================================
    # H0 Persistence Image
    # 논문: birth_range=(0, 1), pers_range=(0, max_eps), skew=False
    # =========================================================================
    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 1)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.array(barcodes[0])
    if len(bars_h0) > 0:
        # H0는 skew=False (이미 birth-persistence 형태로 간주)
        img_h0 = pers_imager_h0.transform(bars_h0, skew=False)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))

    # 논문: np.mean(img, axis=0) - birth 축을 따라 평균
    img0_1d = np.mean(img_h0, axis=0)

    # =========================================================================
    # H1 Persistence Image
    # 논문: birth_range=(0, max_eps), pers_range=(0, max_eps/2), skew=True
    # =========================================================================
    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.array(barcodes[1])
    if len(bars_h1) > 0:
        # H1은 skew=True ((birth, death) → (birth, persistence) 변환)
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))

    # =========================================================================
    # Normalization & Output
    # =========================================================================
    if normalization:
        if np.max(img0_1d) > 0:
            vector[0] = img0_1d / np.max(img0_1d)
        else:
            vector[0] = img0_1d

        if np.max(img_h1) > 0:
            vector[1] = img_h1.flatten() / np.max(img_h1)
        else:
            vector[1] = img_h1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img_h1.flatten()

    return vector

def visualize_PIs(PIs, max_eps=10, px_res=0.1):
    import matplotlib.pyplot as plt

    # H0: shape (100,) - 1D image
    # 논문에서 mean(axis=0)으로 1D가 되었으므로 그대로 plot
    h0_img = PIs[0]
    h0_length = int(max_eps / px_res)  # 100

    # H1: shape (5000,) → reshape to (100, 50)
    h1_birth_bins = int(max_eps / px_res)  # 100
    h1_pers_bins = int((max_eps / 2) / px_res)  # 50
    h1_img = PIs[1].reshape((h1_birth_bins, h1_pers_bins))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # H0 (1D plot)
    axes[0].plot(h0_img, 'b-', linewidth=1.5)
    axes[0].fill_between(range(len(h0_img)), h0_img, alpha=0.3)
    axes[0].set_title(f'H0 Persistence Image (1D)\nLength: {h0_length}')
    axes[0].set_xlabel('Persistence (Bin Index)')
    axes[0].set_ylabel('Intensity')
    axes[0].set_xlim([0, h0_length])
    axes[0].grid(True, alpha=0.3)

    # H1 (2D image)
    im = axes[1].imshow(h1_img.T, cmap='hot', origin='lower', aspect='auto')
    axes[1].set_title(f'H1 Persistence Image (2D)\nResolution: ({h1_birth_bins}, {h1_pers_bins})')
    axes[1].set_xlabel('Birth (Bin Index)')
    axes[1].set_ylabel('Persistence (Bin Index)')
    plt.colorbar(im, ax=axes[1], label='Intensity')

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_all_barcodes(A, B, max_edge=10):
    """
    Compute Image, Kernel, Cokernel barcodes for the inclusion L ⊆ K.
    L = Rips(A), K = Rips(A ∪ B)
    """
    total = np.concatenate([A, B], axis=0)
    a = len(A)

    # Build Rips complex
    st = compute_Rips(total, max_edge)
    simplices, filt = divide_filtration(st)
    m = len(simplices)

    # Identify simplices in L vs K-L
    in_L = [all(v < a for v in s) for s in simplices]
    idx_L = [i for i, b in enumerate(in_L) if b]
    idx_KmL = [i for i, b in enumerate(in_L) if not b]
    set_idx_KmL = set(idx_KmL)
    g2L = {g: pos for pos, g in enumerate(idx_L)}

    # Step 1: Reduce Df and Dg
    Df = _build_boundary(simplices)
    Rf, lowf, Vf = _reduce_with_V(Df)

    boundary_L = []
    for g_idx in idx_L:
        col = {g2L[r] for r in Df[g_idx] if r in g2L}
        boundary_L.append(col)
    Rg, lowg, Vg = _reduce_with_V(boundary_L)

    # Step 2: Build Dim and reduce
    row_order = idx_L + idx_KmL
    row_remap = {g: i for i, g in enumerate(row_order)}
    inv_row_remap = {i: g for g, i in row_remap.items()}

    Dim = [{row_remap[r] for r in Df[col_idx]} for col_idx in range(m)]
    Rim, lowim, _ = _reduce_with_V(Dim)

    # Build Vim by remapping Vf
    Vim = [{row_remap[r] for r in Vf[col_idx]} for col_idx in range(m)]

    # Step 3: Build Dker and reduce
    cycle_cols = [i for i in range(m) if not Rim[i]]
    Dker = [Vim[c] for c in cycle_cols]
    if Dker:
        _, lowker, _ = _reduce_with_V(Dker)
    else:
        lowker = []
    cycle_pos = {c: pos for pos, c in enumerate(cycle_cols)}

    # Step 4: Build Dcok and reduce
    Dcok = []
    for i in range(m):
        if in_L[i]:
            jL = g2L[i]
            if not Rg[jL]:
                Dcok.append({idx_L[pos] for pos in Vg[jL]})
                continue
        Dcok.append(set(Df[i]))
    _, lowcok, _ = _reduce_with_V(Dcok)

    # Format output helper
    def _format_output(bars_dict):
        out = {}
        # 항상 0, 1 dimension을 포함하도록 보장
        for p in [0, 1]:
            if p in bars_dict and bars_dict[p]:
                arr = np.array(bars_dict[p])
                out[p] = arr[np.lexsort((arr[:, 1], arr[:, 0]))]
            else:
                out[p] = np.empty((0, 2))
        return out

    # Image Barcode
    image_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        sigma = inv_row_remap[lowim[tau]]
        if sigma in g2L:
            birth_val, death_val = filt[sigma], filt[tau]
            if birth_val != death_val:
                p = len(simplices[sigma]) - 1
                if p >= 0:
                    image_bars[p].append((birth_val, death_val))

    # Kernel Barcode
    kernel_bars = defaultdict(list)
    for tau in idx_L:
        jL = g2L[tau]
        if not Rg[jL] or Rf[tau] or tau not in cycle_pos:
            continue
        local_col = cycle_pos[tau]
        if local_col >= len(lowker):
            continue
        low_local = lowker[local_col]
        if low_local == -1:
            continue
        sigma = inv_row_remap[low_local]
        if in_L[sigma]:
            continue
        birth_val, death_val = filt[sigma], filt[tau]
        if birth_val != death_val:
            p = len(simplices[sigma]) - 2
            if p >= 0:
                kernel_bars[p].append((birth_val, death_val))

    # Cokernel Barcode
    cok_bars = defaultdict(list)
    for tau in range(m):
        if not Rf[tau] or lowim[tau] == -1:
            continue
        row_global = inv_row_remap[lowim[tau]]
        if row_global not in set_idx_KmL:
            continue
        lowc = lowcok[tau]
        if lowc == -1:
            continue
        sigma = lowc
        birth_val, death_val = filt[sigma], filt[tau]
        if birth_val != death_val:
            p = len(simplices[sigma]) - 1
            if p >= 0:
                cok_bars[p].append((birth_val, death_val))

    return {
        'image': _format_output(image_bars),
        'kernel': _format_output(kernel_bars),
        'cokernel': _format_output(cok_bars)
    }

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/URP/1224_Vectors"
BASE_DIR1 = "/content/drive/MyDrive/URP"
OUT_DIR_1  = os.path.join(BASE_DIR, "Sixpack_Rips")
#OUT_DIR_2 = os.path.join(BASE_DIR, "Sixpack_Chroma")
os.makedirs(OUT_DIR_1, exist_ok=True)

A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

In [ ]:
import time
for idx in range(28, 513):
    print(f"[{idx}] 실행 시작")
    start_time = time.time()   # 시작 시간 기록
    params = PARAM_LIST[idx - 1]
    folder = os.path.join(BASE_DIR1, f"ParamSweep_{idx}_Output")
    pos_file_path   = os.path.join(folder, f"Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat")
    types_file_path = os.path.join(folder, f"Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat")
    save_path_1  = os.path.join(OUT_DIR_1, f"Sixpack_Rips_{idx}.npz")
    #save_path_2  = os.path.join(OUT_DIR_2, f"Sixpack_Chroma_{idx}.npz")
    types = np.loadtxt(types_file_path, dtype=int)
    positions = np.loadtxt(pos_file_path, delimiter=',')
    A = positions[types == 1]
    B = positions[types == 2]
    six_pack_A_to_B = compute_all_barcodes(A, B)
    six_pack_B_to_A = compute_all_barcodes(B, A)
    PB_total = compute_Persistence_barcode(np.concatenate([A, B], axis=0))
    PB_A = compute_Persistence_barcode(A)
    PB_B = compute_Persistence_barcode(B)
    six_pack_A_to_B.update({'complex': PB_total, 'sub_complex': PB_A, 'relative' : PB_B})
    six_pack_B_to_A.update({'complex': PB_total, 'sub_complex': PB_B, 'relative' : PB_A})
    PI_six_pack_A_to_B = {}
    PI_six_pack_B_to_A = {}

    for key in six_pack_A_to_B.keys():
        PI = compute_PIs(six_pack_A_to_B[key], normalization = False)
        PI_six_pack_A_to_B[key] = PI


    for key in six_pack_B_to_A.keys():
        PI = compute_PIs(six_pack_B_to_A[key], normalization = False)
        PI_six_pack_B_to_A[key] = PI
    np.savez_compressed(save_path_1,
                            PI_six_pack_A_to_B,
                            PI_six_pack_B_to_A)
    end_time = time.time()     # 종료 시간 기록
    elapsed = end_time - start_time
    print(f"[{idx}] 실행 완료 (걸린 시간: {elapsed:.2f} 초)")


[28] 실행 시작
[28] 실행 완료 (걸린 시간: 190.34 초)
[29] 실행 시작
[29] 실행 완료 (걸린 시간: 60.39 초)
[30] 실행 시작
[30] 실행 완료 (걸린 시간: 60.96 초)
[31] 실행 시작
[31] 실행 완료 (걸린 시간: 81.52 초)
[32] 실행 시작
[32] 실행 완료 (걸린 시간: 179.98 초)
[33] 실행 시작
[33] 실행 완료 (걸린 시간: 97.06 초)
[34] 실행 시작
[34] 실행 완료 (걸린 시간: 327.91 초)
[35] 실행 시작
[35] 실행 완료 (걸린 시간: 310.74 초)
[36] 실행 시작
[36] 실행 완료 (걸린 시간: 287.34 초)
[37] 실행 시작
[37] 실행 완료 (걸린 시간: 147.81 초)
[38] 실행 시작
[38] 실행 완료 (걸린 시간: 314.61 초)
[39] 실행 시작
[39] 실행 완료 (걸린 시간: 151.77 초)
[40] 실행 시작
[40] 실행 완료 (걸린 시간: 316.02 초)
[41] 실행 시작
[41] 실행 완료 (걸린 시간: 159.36 초)
[42] 실행 시작
[42] 실행 완료 (걸린 시간: 128.40 초)
[43] 실행 시작
[43] 실행 완료 (걸린 시간: 215.99 초)
[44] 실행 시작
[44] 실행 완료 (걸린 시간: 336.45 초)
[45] 실행 시작
[45] 실행 완료 (걸린 시간: 307.62 초)
[46] 실행 시작
[46] 실행 완료 (걸린 시간: 265.22 초)
[47] 실행 시작
[47] 실행 완료 (걸린 시간: 300.76 초)
[48] 실행 시작
[48] 실행 완료 (걸린 시간: 136.77 초)
[49] 실행 시작
[49] 실행 완료 (걸린 시간: 205.82 초)
[50] 실행 시작
[50] 실행 완료 (걸린 시간: 229.53 초)
[51] 실행 시작
[51] 실행 완료 (걸린 시간: 259.88 초)
[52] 실행 시작
[52] 실행 완료 (걸린 시간: 333.85 초)
[53]